# 14 — Ax Bayesian optimization for joint arrival calibration (T-163)

Joint search over **p_short**, **q10**, and **delta_c** for the abdella_mix corridor.

Rust-first via ``evaluate_joint_calib_trial_py`` (``joint_arrival_calib::evaluate_fast_trial``).

**Fast metrics** (~15–25s/trial here): ac2_19 min margin (hard reject if ≤0), session f (seed 163503), truth band p50 + pct_60_90 (400 MC draws).

**Slow metric** (FULL_RUN): ac2_11a_ratio mean+SEM over K BO seeds.

Defaults are smoke-sized. Set ``FULL_RUN = True`` for 60 trials with ac2_11a objective leg.

Legacy grid examples are diagnostic only.

## Setup

```bash
uv run maturin develop --release -m crates/voi_py/Cargo.toml
uv sync --extra notebooks
uv run jupyter lab
```

CLI: ``uv run python scripts/run_arrival_calib_bo.py --smoke``


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import numpy as np
from ax.api.client import Client
from tqdm.auto import tqdm

from blueberries_voi.backend import rust_available, rust_core
from blueberries_voi.experiments.arrival_joint_calib import (
    REJECTED_OBJECTIVE,
    ax_outcome_constraints,
    ax_parameter_configs,
    benchmark_joint_calib_trial,
    evaluate_with_replicates,
)

os.environ.setdefault("OMP_NUM_THREADS", "1")
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "blueberries_voi").is_dir():
    REPO_ROOT = REPO_ROOT.parent

FULL_RUN = False
if FULL_RUN:
    K_BO_SEEDS, TOTAL_AX_TRIALS, AX_PARALLELISM, INCLUDE_AC2_11A = 6, 60, 4, True
else:
    K_BO_SEEDS, TOTAL_AX_TRIALS, AX_PARALLELISM, INCLUDE_AC2_11A = 2, 10, 2, False

EXTRA_AX_TRIALS = 0
RELOAD_AX = False
RNG = np.random.default_rng(20260828)
BO_SEEDS = [int(RNG.integers(0, 2**31 - 1)) for _ in range(K_BO_SEEDS)]
AX_JSON = REPO_ROOT / "outputs" / "arrival_joint_calib_bo_ax_client.json"
OUTPUT_JSON = REPO_ROOT / "outputs" / "arrival_joint_calib_bo.json"

rust_fn = getattr(rust_core, "evaluate_joint_calib_trial_py", None) if rust_core else None
print(f"Rust: {rust_available() and rust_fn is not None}, FULL_RUN={FULL_RUN}")
print(f"trials={TOTAL_AX_TRIALS} K={K_BO_SEEDS} slow={INCLUDE_AC2_11A}")


In [ ]:
elapsed = benchmark_joint_calib_trial()
print(json.dumps({"fast_metrics_s": elapsed}, indent=2))


In [ ]:
def _completed(ax_client: Client) -> int:
    return sum(1 for t in ax_client._experiment.trials.values() if t.status.is_completed)

if RELOAD_AX and AX_JSON.is_file():
    client = Client.load_from_json_file(str(AX_JSON))
    trials_to_run = max(0, TOTAL_AX_TRIALS - _completed(client))
    trial_log: list[dict[str, Any]] = []
else:
    client = Client()
    client.configure_experiment(name="t163-joint-arrival-calib", parameters=ax_parameter_configs())
    client.configure_optimization(objective="ac2_11a_ratio", outcome_constraints=ax_outcome_constraints())
    trial_log = []
    trials_to_run = TOTAL_AX_TRIALS

completed = 0
pbar = tqdm(total=trials_to_run, desc="Ax arrival joint BO")
while completed < trials_to_run:
    batch_n = min(AX_PARALLELISM, trials_to_run - completed)
    trials = client.get_next_trials(max_trials=batch_n)
    for trial_index, parameters in trials.items():
        metrics = evaluate_with_replicates(
            float(parameters["p_short"]),
            float(parameters["q10"]),
            float(parameters["delta_c"]),
            BO_SEEDS,
            include_ac2_11a=INCLUDE_AC2_11A,
        )
        client.complete_trial(
            trial_index=trial_index,
            raw_data={
                "ac2_11a_ratio": metrics["ac2_11a_ratio"],
                "session_f": metrics["session_f"],
                "p50": metrics["p50"],
                "pct_60_90": metrics["pct_60_90"],
            },
        )
        trial_log.append({
            "trial_index": int(trial_index),
            "p_short": float(parameters["p_short"]),
            "q10": float(parameters["q10"]),
            "delta_c": float(parameters["delta_c"]),
            "rejected": metrics["ac2_11a_ratio"][0] <= REJECTED_OBJECTIVE / 2,
            "mean_ac2_11a": metrics["ac2_11a_ratio"][0],
            "session_f": metrics["session_f"][0],
            "p50": metrics["p50"][0],
            "pct_60_90": metrics["pct_60_90"][0],
        })
    AX_JSON.parent.mkdir(parents=True, exist_ok=True)
    client.save_to_json_file(str(AX_JSON))
    completed += len(trials)
    pbar.update(len(trials))
pbar.close()
best_params, _, best_index, _ = client.get_best_parameterization()
print(f"Best trial {best_index}: {best_params}")


In [ ]:
payload = {
    "full_run": FULL_RUN,
    "include_ac2_11a": INCLUDE_AC2_11A,
    "bo_seeds": BO_SEEDS,
    "trials": trial_log,
    "best_parameters": dict(best_params),
}
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {OUTPUT_JSON}")
